In [7]:
import kagglehub                #para importar datos del desafio
from google.colab import files  #para guardar datos en Drive (características y tokens)

from sklearn.cluster import KMeans    #para la clusterización en la función de tokenización
from sklearn.metrics import f1_score  #para utilizar la métrica de rendimiento f1_score
from sklearn.model_selection import train_test_split    #para la división de datos en conjuntos de entrenamiento y test

import pandas as pd             #para la manipulación de datos
import numpy as np              #para la manipulación de arreglos
import os                       #para operar con los archivos y directorios
import librosa                  #para el procesamiento de señales de audio
import librosa.display          #para visualizar de forma gráfica características de audio
import tensorflow as tf         #para trabajar con redes y tensores

#Elementos de tensorflow
from tensorflow.keras.utils import to_categorical       #para la converción de datos categórico (one hot)
from tensorflow.keras.layers import Input               #para definir la forma de los datos de entrada
from tensorflow.keras.layers import Dense               #para crear y definir la capa densa
from tensorflow.keras.layers import Dropout             #para definir la técnica de regularización dropout
from tensorflow.keras.layers import Embedding           #para la conversión de categorías en vectores densos de tamaño fijo
from tensorflow.keras.layers import GlobalMaxPooling1D  #para realizar la reducción (pooling) en datos temporales o secuenciales
from tensorflow.keras.models import Sequential          #para inicializar una pila lineal de capas
from tensorflow.keras.callbacks import EarlyStopping    #para definir criterios de detección temprana
from tensorflow.keras import regularizers               #para definir regulizadores L1 L2
from tensorflow.keras.preprocessing.sequence import pad_sequences   #para normalizar la longitud de listas de secuencias, transformándolas en un arreglo rectangular uniforme


In [8]:
# ==========================================
# Funciones de Extracción de Características
# ==========================================
def obtener_caracteristicas_tradicionales(ruta_archivo):
    """Extrae la matriz de MFCCs estándar de un audio."""
    y, sr = librosa.load(ruta_archivo)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return mfccs

In [9]:
# ==========================================
# Funciones de Modelado Matemático y Métricas
# ==========================================
def calculate_f1(y_true, y_pred_probs):
    """Calcula el F1-score macro procesando la matriz probabilística 2D."""
    y_pred = np.argmax(y_pred_probs, axis=1)
    return f1_score(y_true, y_pred, average='macro')

In [10]:
def train_evaluate_model(activation, depth, neurons, learning_rate, optimizer_name='SGD',
                         batch_size=32, initializer='glorot_uniform', dropout_rate=0,
                         regularizer_type=None, regularizer_lambda=0, epochs=200, patience=50,
                         vocab_size=100, max_length=500):

    if regularizer_type == 'l1':
        reg = regularizers.l1(regularizer_lambda)
    elif regularizer_type == 'l2':
        reg = regularizers.l2(regularizer_lambda)
    else:
        reg = None

    model = Sequential()

    # Entrada compatible con los estándares de Keras 3
    model.add(Input(shape=(max_length,), dtype='int32'))
    model.add(Embedding(input_dim=vocab_size, output_dim=64))
    model.add(GlobalMaxPooling1D())

    # Primera capa oculta
    model.add(Dense(neurons, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg))
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))

    # Capas ocultas adicionales según 'depth'
    for _ in range(depth - 1):
        model.add(Dense(neurons, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    # Salida Softmax fija para las 8 clases de estilos musicales
    model.add(Dense(8, activation='softmax'))

    # Selección de optimizador geométrico
    if optimizer_name == 'SGD':
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == 'Adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'RMSprop':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    # El entrenamiento se realiza directamente sobre los tokens nativos estables
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    y_val_pred_probs = model.predict(X_val, verbose=0)
    y_test_pred_probs = model.predict(X_test, verbose=0)

    val_f1 = calculate_f1(y_val, y_val_pred_probs)
    test_f1 = calculate_f1(y_test, y_test_pred_probs)

    return history, val_f1, test_f1, model

In [ ]:
# ==========================================
# 1. Configuración de Entorno y Descarga
# ==========================================
print("--- Paso 1: Configurando Kaggle y Google Drive ---")
files.upload() # Sube tu archivo kaggle.json aquí

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

path = kagglehub.competition_download('clasificacion-de-generos-musicales')
print("Ruta de los archivos de la competencia:", path)

drive.mount('/content/drive')

# Seteo de rutas del pipeline
ruta_audio_train = '/root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales/train'
ruta_salida_base = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Desafio_2/Train'
carpeta_mfccs = os.path.join(ruta_salida_base, 'mfccs/')
carpeta_tokens_mfcc = os.path.join(ruta_salida_base, 'tokens/tokens_mfcc/')

os.makedirs(carpeta_mfccs, exist_ok=True)
os.makedirs(carpeta_tokens_mfcc, exist_ok=True)



# Procesamiento y guardado inicial de MFCCs (Si ya los tienes guardados, puedes saltar este bucle)
print("\n--- Paso 2: Extrayendo y guardando matrices MFCC originales en disco ---")
errores = []

for raiz, directorios, archivos in os.walk(ruta_audio_train):
    for archivo in archivos:
        if not archivo.endswith('.mp3'):
            continue
        ruta_completa = os.path.join(raiz, archivo)
        try:
            feature_mfccs = obtener_caracteristicas_tradicionales(ruta_completa)
            # Guardamos la matriz MFCC cruda (13, ventanas)
            np.save(os.path.join(carpeta_mfccs, f"{os.path.splitext(archivo)[0]}_mfccs_entrenamiento.npy"), feature_mfccs)
        except Exception as e:
            print(f"Error procesando {archivo}: {e}")
            errores.append(archivo)

if errores:
    with open(os.path.join(ruta_salida_base, 'errores.txt'), 'w', encoding='utf-8') as f:
        f.write("\n".join(errores))

# ==========================================
# 3. Creación del Codebook Global (Vector Quantization)
# ==========================================
print("\n--- Paso 3: Entrenando el Codebook Global (MiniBatchKMeans) ---")
archivos_mfcc = [os.path.join(carpeta_mfccs, f) for f in os.listdir(carpeta_mfccs) if f.endswith('.npy')]

# Tomamos una muestra aleatoria de 300 canciones para armar el vocabulario sin saturar la RAM
np.random.seed(42)
archivos_muestra = np.random.choice(archivos_mfcc, size=min(300, len(archivos_mfcc)), replace=False)

muestra_para_fit = []
for ruta_npy in archivos_muestra:
    mfcc_cancion = np.load(ruta_npy) # (13, ventanas)
    muestra_para_fit.append(mfcc_cancion.T) # Transponemos a (ventanas, 13)

X_codebook_train = np.vstack(muestra_para_fit)
print(f"Dimensiones de la matriz para entrenar el vocabulario: {X_codebook_train.shape}")

# Entrenamos el cuantizador global de 100 centroides
kmeans_global = MiniBatchKMeans(n_clusters=100, random_state=42, batch_size=2048)
kmeans_global.fit(X_codebook_train)
print("✅ Codebook Global unificado entrenado con éxito.")

# ==========================================
# 4. Tokenización Coherente
# ==========================================
print("\n--- Paso 4: Cuantizando audios con el vocabulario universal ---")
for archivo_name in os.listdir(carpeta_mfccs):
    if not archivo_name.endswith('.npy'):
        continue

    ruta_completa_mfcc = os.path.join(carpeta_mfccs, archivo_name)
    mfcc_cancion = np.load(ruta_completa_mfcc) # (13, ventanas)

    # Predecir los índices de clusters usando el modelo global unificado
    tokens_consistentes = kmeans_global.predict(mfcc_cancion.T)

    # Nombre estructurado idéntico al que busca tu pipeline de carga
    nombre_guardado = archivo_name.replace('_mfccs_entrenamiento.npy', '_mfccs_entrenamiento_tokens_mfccs_entrenamiento.npy')
    np.save(os.path.join(carpeta_tokens_mfcc, nombre_guardado), tokens_consistentes)

print("Todos los archivos de tokens han sido regenerados con coherencia semántica.")

# ==========================================
# 5. Carga y Alineación Definitiva de Datos
# ==========================================
print("\n--- Paso 5: Cargando y Sincronizando Dataset con el CSV ---")
df_train = pd.read_csv('train.csv')

tokens_filtrados = []
labels_filtradas = []
descartados = 0

for index, row in df_train.iterrows():
    nombre_base = row['filename']
    etiqueta_numerica = row['label']

    nombre_archivo_npy = f"{os.path.splitext(nombre_base)[0]}_mfccs_entrenamiento_tokens_mfccs_entrenamiento.npy"
    ruta_completa_token = os.path.join(carpeta_tokens_mfcc, nombre_archivo_npy)

    if not os.path.exists(ruta_completa_token):
        descartados += 1
        continue

    secuencia_tokens = np.load(ruta_completa_token)
    tokens_filtrados.append(secuencia_tokens)
    labels_filtradas.append(int(etiqueta_numerica))

print(f"Canciones alineadas con éxito: {len(tokens_filtrados)}")
print(f"Archivos omitidos/no encontrados: {descartados}")

# Padding para Keras
MAX_LEN = 500
X_final = pad_sequences(tokens_filtrados, maxlen=MAX_LEN, padding='post', truncating='post')
y_final = np.array(labels_filtradas) # Vector plano de enteros para sparse_categorical_crossentropy

# Partición Estratificada de Datos
X_temp, X_test, y_temp, y_test = train_test_split(
    X_final, y_final, test_size=0.1, stratify=y_final, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2/0.9, stratify=y_temp, random_state=42
)

print(f"Estructuras listas: Train {X_train.shape}, Validation {X_val.shape}, Test {X_test.shape}")





# ==========================================
# 7. Ejecución del Random Search
# ==========================================
print("\n--- Paso 6: Iniciando Optimización por Random Search ---")
activations_list  = ['relu', 'sigmoid', 'tanh']
depths            = [1, 2, 3]
neurons_list      = [20, 50, 100]
learning_rates    = [0.001, 0.01, 0.1]

fixed_batch_size  = 32
fixed_initializer = 'glorot_uniform'
fixed_optimizer   = 'SGD'
fixed_patience    = 50

num_trials = 20
results = []
best_f1 = -1
best_config = None

for i in range(num_trials):
    activation = random.choice(activations_list)
    depth = random.choice(depths)
    neurons = random.choice(neurons_list)
    lr = random.choice(learning_rates)

    print(f"\nTrial {i+1}/{num_trials} -> Config: act={activation}, depth={depth}, neurons={neurons}, lr={lr}")

    history, val_f1, test_f1, model = train_evaluate_model(
        activation=activation, depth=depth, neurons=neurons, learning_rate=lr,
        optimizer_name=fixed_optimizer, batch_size=fixed_batch_size,
        initializer=fixed_initializer, epochs=200, patience=fixed_patience
    )

    results.append({
        'activation': activation, 'depth': depth, 'neurons': neurons,
        'learning_rate': lr, 'val_f1': val_f1, 'test_f1': test_f1
    })

    print(f"Resultados del Trial -> Val F1: {val_f1:.4f} | Test F1: {test_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = results[-1]
        best_history = history
        best_model_keras = model
        print("🔥 ¡Nuevo mejor modelo guardado!")

print("\n===== PROCESO COMPLETADO: MEJOR CONFIGURACIÓN ENCONTRADA =====")
print(best_config)

--- Paso 1: Configurando Kaggle y Google Drive ---


Saving kaggle.json to kaggle (2).json
Ruta de los archivos de la competencia: /root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Paso 2: Extrayendo y guardando matrices MFCC originales en disco ---


/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error procesando 133297.mp3: 


/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184:

Error procesando 099134.mp3: 


/tmp/ipykernel_1470/263440986.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(ruta_archivo)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error procesando 108925.mp3: 

--- Paso 3: Entrenando el Codebook Global (MiniBatchKMeans) ---
Dimensiones de la matriz para entrenar el vocabulario: (387570, 13)
✅ Codebook Global unificado entrenado con éxito.

--- Paso 4: Cuantizando audios con el vocabulario universal ---
Todos los archivos de tokens han sido regenerados con coherencia semántica.

--- Paso 5: Cargando y Sincronizando Dataset con el CSV ---
Canciones alineadas con éxito: 6267
Archivos omitidos/no encontrados: 3
Estructuras listas: Train (4386, 500), Validation (1254, 500), Test (627, 500)

--- Paso 6: Iniciando Optimización por Random Search ---

Trial 1/20 -> Config: act=relu, depth=1, neurons=20, lr=0.1
Resultados del Trial -> Val F1: 0.3842 | Test F1: 0.4032
🔥 ¡Nuevo mejor modelo guardado!

Trial 2/20 -> Config: act=tanh, depth=3, neurons=20, lr=0.1
Resultados del Trial -> Val F1: 0.3903 | Test F1: 0.3821
🔥 ¡Nuevo mejor modelo guardado!

Trial 3/20 -> Config: act=tanh, depth=2, neurons=100, lr=0.001
Resultados de

In [ ]:
# =====================================================================
# 8. PIPELINE DE INFERENCIA: PROCESAR TEST Y GENERAR SUBMISSION KAGGLE
# =====================================================================
import os
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("--- Iniciando el proceso de etiquetado para el conjunto de Test ---")

# 1. Cargar el archivo de metadatos de prueba (Asegúrate de que el nombre sea el correcto en tu entorno)
df_test = pd.read_csv('test.csv')

# 2. Definir las rutas de las carpetas de Test en tu Google Drive
# NOTA: Asegúrate de que estas rutas apanten exactamente a donde guardaste los archivos de Test
carpeta_mfccs_test = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Test/mfccs/'
carpeta_tokens_test = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Test/tokens/tokens_mfcc/'
os.makedirs(carpeta_tokens_test, exist_ok=True)

tokens_test_filtrados = []
indices_validos_test = []
descartados_test = 0

print("Fase A: Cuantizando y alineando archivos de prueba con el Codebook Global...")

# Recorremos el CSV de prueba fila por fila
for index, row in df_test.iterrows():
    nombre_base = row['filename']

    # Construimos el nombre del archivo de características .npy esperado para test
    # (Ajusta la extensión según cómo se hayan guardado tus npy de test en los pasos previos)
    nombre_archivo_mfcc = f"{os.path.splitext(nombre_base)[0]}_mfccs_entrenamiento.npy"
    ruta_completa_mfcc = os.path.join(carpeta_mfccs_test, nombre_archivo_mfcc)

    # Si la matriz MFCC de la canción de prueba no existe, la saltamos de forma segura
    if not os.path.exists(ruta_completa_mfcc):
        descartados_test += 1
        continue

    # 3. Cargar la matriz MFCC original (13, ventanas)
    mfcc_cancion = np.load(ruta_completa_mfcc)

    # 4. REGLA DE ORO DE MIR: Cuantizar usando estrictamente el K-Means global del entrenamiento (SIN .fit)
    tokens_consistentes_test = kmeans_global.predict(mfcc_cancion.T)

    # Guardamos el archivo de tokens transformados en el disco por seguridad
    nombre_archivo_token = nombre_archivo_mfcc.replace('_mfccs_entrenamiento.npy', '_mfccs_entrenamiento_tokens_mfccs_entrenamiento.npy')
    np.save(os.path.join(carpeta_tokens_test, nombre_archivo_token), tokens_consistentes_test)

    # Almacenamos la secuencia de tokens en memoria y registramos su índice del CSV original
    tokens_test_filtrados.append(tokens_consistentes_test)
    indices_validos_test.append(index)

print(f"\n--- Resumen de la Fase A ---")
print(f"Canciones de prueba procesadas con éxito: {len(tokens_test_filtrados)}")
print(f"Archivos .npy no encontrados en la carpeta de Test: {descartados_test}")

# 5. Formato final con el mismo Padding exacto del entrenamiento
MAX_LEN = 500
X_test_unlabeled = pad_sequences(tokens_test_filtrados, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"\nTamaño final del tensor de prueba para la red: {X_test_unlabeled.shape}")

# 6. Ejecutar predicciones utilizando la variable que contiene el MEJOR MODELO del Random Search
print("\nFase B: Ejecutando inferencia con el mejor modelo guardado...")
predicciones_probs = best_model_keras.predict(X_test_unlabeled, verbose=1)

# Extraemos el índice de la clase con mayor probabilidad (0 al 7)
clases_predichas = np.argmax(predicciones_probs, axis=1)

# 7. Traducir los índices numéricos a las etiquetas de texto originales de Kaggle
# Usamos el diccionario cargado en los pasos anteriores para revertir el mapeo
# Como el diccionario tiene la estructura {texto: numero}, lo invertimos a {numero: texto}
inverso_estilos_dict = {v: k for k, v in estilos_dict.items()}

# Creamos un DataFrame copia que contenga únicamente las filas sincronizadas que sí tenían archivo
submission = df_test.iloc[indices_validos_test].copy()

# Asignamos las etiquetas predichas mapeándolas con el diccionario invertido
submission['label'] = pd.Series(clases_predichas, index=submission.index).map(inverso_estilos_dict)

# 8. Asegurar el formato exacto requerido por Kaggle (Filtrar solo columnas filename y label)
submission_final = submission[['filename', 'label']]

# Exportar a un archivo CSV en el directorio local de Colab
submission_final.to_csv('submission_kaggle.csv', index=False)

print("\n🎉 ¡Proceso Finalizado con Éxito! 🎉")
print("El archivo 'submission_kaggle.csv' ha sido generado y está listo para ser descargado y subido a Kaggle.")
print("\nPrimeras filas del archivo a enviar:")
print(submission_final.head(10))